Carregando arquivo para leitura


In [9]:
#Arquivos de sintomas relatados
with open('../docs/Frases_Sintomas_Faceis.txt', 'r', encoding='utf-8') as arquivo:
    for linha in arquivo:
        print(linha.strip())


"Estou com dor no peito e suor frio desde cedo."
"Senti um aperto no peito, fiquei pálido e com a mão formigando."
"Tenho suor frio e náusea há uma hora."
"Sinto dor no ombro e pescoço, com suor frio."
"Sinto falta de ar e dor nas costas."
"Tenho dor ao fazer esforço, mas passa quando eu descanso."
"Estou com falta de ar e tenho alívio em repouso."
"Sinto falta de ar, dor nas costas e alívio em repouso."
"Sinto palpitação e tontura."
"Tenho palpitação e visão preta de repente."
"Meu coração fica acelerado e fico com tontura."
"Tive palpitação, tontura e visão preta."
"Tenho hipertensão e falta de ar."
"Sinto falta de ar ao deitar e edema nos pés."
"Fico com falta de ar ao se esforçar e inchaço na perna."
"Tenho falta de ar ao se esforçar e falta de ar ao deitar."
"Estou com dor de barriga e febre."
"Tenho dor no abdomem e muitos gases."
"Sinto tontura e estou com glicemia baixa."
"Bati o peito e sinto dor quando respiro."


Carregando arquivo CSV com rotulação de diagnosticos

In [10]:
import pandas as pd

df = pd.read_csv('../docs/sintomas.csv', encoding="utf-8")


In [11]:
df.head(5)

,sintomas_1,sintomas_2,doenca_associada,sigla_doenca
0,"""dor no ombro e pescoço""","""suor frio""",infarto,MI
1,"""falta de ar ao se esforçar""","""suor frio""",Infarto,MI
2,"""suor em repouso""","""dor no peito""",infarto,MI
3,"""palido""","""aperto no peito""",infarto,MI
4,"""mão formigando""","""aperto no peito""",infarto,MI


Sugestão de diagnostico utilizando ONTOLOGIA

In [13]:
import csv
import re
import unicodedata
from collections import defaultdict
from pathlib import Path

ARQ_FRASES = "../docs/Frases_Sintomas_Faceis.txt"
ARQ_MAPA = "../docs/sintomas.csv"
ARQ_SAIDA = "../docs/resultado_diagnosticos.csv"

def normalizar(texto: str) -> str:
    #Trata o texto normalizando a escrita
    texto = unicodedata.normalize("NFD", texto.lower())
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    texto = re.sub(r"[^\w\s]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()

#Carrega cada frase do texto
def carregar_frases(caminho: Path) -> list[str]:
    with open(caminho, encoding="utf-8") as f:
        return [linha.strip() for linha in f if linha.strip()]

#Carrega a tabela com rotulos e juntam as duas colunas de sintomas , que servem como base para a analise das frases
def carregar_ontologia(caminho: Path) -> dict[str, set[str]]:

    ontologia = defaultdict(set)
    with open(caminho, encoding="utf-8", newline="") as f:
        for linha in csv.DictReader(f):
            doenca = linha["doenca_associada"].strip()
            for coluna in ("sintomas_1", "sintomas_2"):
                expressao = normalizar(linha[coluna])
                if expressao:
                    ontologia[doenca].add(expressao)
    return ontologia

def analisar_frase(frase: str, ontologia: dict[str, set[str]]) -> list[tuple[str, list[str]]]:

    #Retorna lista de (doença, sintomas_encontrados) ordenada da maior para a menor quantidade de sintomas.

    texto = f" {normalizar(frase)} "  # espaços garantem correspondência de palavra inteira
    resultados = []
    for doenca, expressoes in ontologia.items():
        encontrados = sorted(e for e in expressoes if f" {e} " in texto)
        if encontrados:
            resultados.append((doenca, encontrados))
    resultados.sort(key=lambda r: len(r[1]), reverse=True)
    return resultados

def main():
    frases = carregar_frases(ARQ_FRASES)
    ontologia = carregar_ontologia(ARQ_MAPA)
    saida = []

    for i, frase in enumerate(frases, start=1):
        print(f"\nPaciente {i}: {frase}")
        resultados = analisar_frase(frase, ontologia)

        if not resultados:
            print("  -> Nenhum sintoma reconhecido no mapa de conhecimento.")
            saida.append([i, frase, "", "", "Sem diagnóstico"])
            continue

        melhor_doenca, melhores_sintomas = resultados[0]
        todos_sintomas = sorted({s for _, ss in resultados for s in ss})
        print(f"  Sintomas identificados: {', '.join(todos_sintomas)}")
        print("  Ranking de possíveis diagnósticos:")
        for doenca, sintomas in resultados[:3]:
            print(f"    - {doenca}: {len(sintomas)} sintoma(s) ({', '.join(sintomas)})")
        print(f"  => Diagnóstico sugerido: {melhor_doenca}")

        saida.append([i, frase, "; ".join(todos_sintomas), len(melhores_sintomas), melhor_doenca])

    with open(ARQ_SAIDA, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["Paciente", "Frase", "Sintomas identificados", "Quantidade de Sintomas", "Diagnóstico sugerido"])
        w.writerows(saida)

    print(f"\nResultados salvos em: {ARQ_SAIDA}")
    print("Aviso: projeto educacional. Não substitui avaliação médica.")

if __name__ == "__main__":
    main()


Paciente 1: "Estou com dor no peito e suor frio desde cedo."
  Sintomas identificados: dor no peito, suor frio
  Ranking de possíveis diagnósticos:
    - infarto: 2 sintoma(s) (dor no peito, suor frio)
    - Infarto: 2 sintoma(s) (dor no peito, suor frio)
  => Diagnóstico sugerido: infarto

Paciente 2: "Senti um aperto no peito, fiquei pálido e com a mão formigando."
  Sintomas identificados: aperto no peito, mao formigando, palido
  Ranking de possíveis diagnósticos:
    - infarto: 3 sintoma(s) (aperto no peito, mao formigando, palido)
  => Diagnóstico sugerido: infarto

Paciente 3: "Tenho suor frio e náusea há uma hora."
  Sintomas identificados: nausea, suor frio
  Ranking de possíveis diagnósticos:
    - Infarto: 2 sintoma(s) (nausea, suor frio)
    - infarto: 1 sintoma(s) (suor frio)
  => Diagnóstico sugerido: Infarto

Paciente 4: "Sinto dor no ombro e pescoço, com suor frio."
  Sintomas identificados: dor no ombro e pescoco, suor frio
  Ranking de possíveis diagnósticos:
    - i

In [ ]:
df_diagnosticos = pd.read_csv("docs/resultado_diagnosticos.csv", encoding="utf-8")
print(df_diagnosticos.head())

# Analisando e prevendo risco de acordo com os sintomas

In [15]:

import pandas as pd

df_frases_riscos = pd.read_csv("../docs/frases_risco.csv", encoding="utf-8")

print(df_frases_riscos.head())



                                              frases        risco
0                   tive um leve incômodo nas costas  baixo risco
1  tenho um aperto forte no peito que vai para o ...   alto risco
2           estou com um pouco de tosse há dois dias  baixo risco
3  estou com suor frio e náusea e a dor no peito ...   alto risco
4  senti uma leve dor de cabeça depois de ficar n...  baixo risco


Separando dados de treino e teste

In [16]:

from sklearn.model_selection import train_test_split

X = df_frases_riscos['frases']
y = df_frases_riscos['risco']
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

Vetorizando o label 'frases' com TF-IDF

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Criando o vetorizador TF-IDF
vectorizer = TfidfVectorizer()

# Ajustando e transformando os documentos
X_train_vectorizer = vectorizer.fit_transform(X_train)
X_test_vectorizer = vectorizer.transform(X_test)



In [18]:
X_train_df = pd.DataFrame(X_train_vectorizer.toarray(), columns=vectorizer.get_feature_names_out())
print(X_train_df.head(5))

   almoço   ao   ar  arroxeados  azia  barriga  cabeça  cadeira  caminhar  \
0     0.0  0.0  0.0         0.0   0.0      0.0     0.0      0.0  0.432746   
1     0.0  0.0  0.0         0.0   0.0      0.0     0.0      0.0  0.000000   
2     0.0  0.0  0.0         0.0   0.0      0.0     0.0      0.0  0.000000   
3     0.0  0.0  0.0         0.0   0.0      0.0     0.0      0.0  0.000000   
4     0.0  0.0  0.0         0.0   0.0      0.0     0.0      0.0  0.000000   

   cansaço  ...       sem  senti      suor      tive  tontura  torto  tosse  \
0      0.0  ...  0.000000    0.0  0.000000  0.337221      0.0    0.0    0.0   
1      0.0  ...  0.000000    0.0  0.000000  0.000000      0.0    0.0    0.0   
2      0.0  ...  0.000000    0.0  0.000000  0.000000      0.0    0.0    0.0   
3      0.0  ...  0.000000    0.0  0.380316  0.000000      0.0    0.0    0.0   
4      0.0  ...  0.372684    0.0  0.000000  0.000000      0.0    0.0    0.0   

         um       uma  visão  
0  0.000000  0.306468    0.0  


Testando modelo RandomForestClassifier

In [19]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


modelo = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42
)


modelo.fit(X_train_vectorizer, y_train)


y_pred = modelo.predict(X_test_vectorizer)


print("Acurácia:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Acurácia: 0.5
              precision    recall  f1-score   support

  alto risco       1.00      0.33      0.50         3
 baixo risco       0.33      1.00      0.50         1

    accuracy                           0.50         4
   macro avg       0.67      0.67      0.50         4
weighted avg       0.83      0.50      0.50         4



Testando modelo SVM

In [20]:
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV

modelo = CalibratedClassifierCV(
    SVC(kernel="rbf", C=1, gamma="scale"),
    method="sigmoid",
    ensemble=False,
    cv=3,
)
modelo.fit(X_train_vectorizer, y_train)

probs = modelo.predict_proba(X_test_vectorizer)[:, 1]
previsao = modelo.predict(X_test_vectorizer)

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, r2_score, brier_score_loss,
)

# Acurácia e F1 usam as classes previstas
acuracia = accuracy_score(y_test, previsao)
f1 = f1_score(y_test, previsao, pos_label="alto risco")

print(f"Acurácia: {acuracia:.2f}")
print(f"F1-score: {f1:.2f}")
print(f"Precisão: {precision_score(y_test, previsao, pos_label='alto risco'):.2f}")
print(f"Recall:   {recall_score(y_test, previsao, pos_label='alto risco'):.2f}")
print(confusion_matrix(y_test, previsao, labels=["alto risco", "baixo risco"]))
print(classification_report(y_test, previsao))


Acurácia: 1.00
F1-score: 1.00
Precisão: 1.00
Recall:   1.00
[[3 0]
 [0 1]]
              precision    recall  f1-score   support

  alto risco       1.00      1.00      1.00         3
 baixo risco       1.00      1.00      1.00         1

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4



Testando modelo KNN

In [21]:

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report


# Modelo KNN
modelo_knn = KNeighborsClassifier(
    n_neighbors=2,   # número de vizinhos
    metric="cosine"  # métrica de distância (pode ser 'euclidean', 'manhattan', etc.)
)

# Treinar o modelo
modelo_knn.fit(X_train_vectorizer, y_train)

# Previsões
y_pred = modelo_knn.predict(X_test_vectorizer)

# Avaliação
print("Acurácia:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Acurácia: 1.0
              precision    recall  f1-score   support

  alto risco       1.00      1.00      1.00         3
 baixo risco       1.00      1.00      1.00         1

    accuracy                           1.00         4
   macro avg       1.00      1.00      1.00         4
weighted avg       1.00      1.00      1.00         4



Testando modelos

In [36]:
sentece = "sinto dor no peito e falta de ar"
sentece_vec = vectorizer.transform([sentece])

y_pred_knn = modelo_knn.predict(sentece_vec)
y_pred_svm = modelo.predict(sentece_vec)
y_pred_random = modelo.predict(sentece_vec)


print("Predições dos modelos treinados:")
print(f"KNN => {y_pred_knn}")
print(f"SVM => {y_pred_svm}")
print(f"RandomForest => {y_pred_random}")


Predições dos modelos treinados:
KNN => ['alto risco']
SVM => ['alto risco']
RandomForest => ['alto risco']
